In [ ]:
# ============================================================
# CÉLULA 1 — CARREGAMENTO DO AGENTE PLANAPP
# ============================================================
#%pip install "mcp"
import importlib
import agent_jupyter

agent_jupyter = importlib.reload(agent_jupyter)

PlanAppAgent = agent_jupyter.PlanAppAgent

In [ ]:
import asyncio
import html

import ipywidgets as widgets
from IPython.display import display

from agent_jupyter import PlanAppAgent


# ============================================================
# TÍTULO
# ============================================================

titulo = widgets.HTML(
    value="""
    <h2 style="margin:0 0 10px 0;">
        🛰️ PlanApp AI — Planejamento de Enlace
    </h2>
    """
)


# ============================================================
# ENTRADA
# ============================================================

entrada = widgets.Textarea(
    placeholder=(
        "Ex.: Analise um enlace entre a Praça da República "
        "e o Largo do Paissandu em São Paulo."
    ),
    layout=widgets.Layout(
        width="100%",
        height="80px"
    )
)


# ============================================================
# BOTÕES
# ============================================================

botao_analisar = widgets.Button(
    description="🚀 Analisar",
    button_style="primary",
    layout=widgets.Layout(
        width="140px"
    )
)

botao_nova = widgets.Button(
    description="🆕 Nova análise",
    layout=widgets.Layout(
        width="140px"
    )
)


# ============================================================
# STATUS
# ============================================================

status = widgets.HTML(
    value=(
        "<b>Status:</b> "
        "Aguardando solicitação."
    )
)

historico_status = widgets.Output(
    layout=widgets.Layout(
        width="100%",
        max_height="220px",
        overflow="auto",
        border="1px solid #ddd",
        padding="8px"
    )
)


# ============================================================
# LOG DE EXECUÇÃO
# ============================================================

log_execucao = widgets.Output(
    layout=widgets.Layout(
        width="100%",
        max_height="250px",
        overflow="auto",
        border="1px solid #ddd",
        padding="8px"
    )
)


# ============================================================
# MAPA
#
# IMPORTANTE:
# Não usamos Output para renderizar o mapa.
# O mapa é inserido diretamente no VBox.
# ============================================================

mapa_output = widgets.VBox(
    children=[],
    layout=widgets.Layout(
        width="100%",
        min_height="0px"
    )
)


# ============================================================
# RESPOSTA
# ============================================================

resposta = widgets.HTML(
    value=""
)


# ============================================================
# CALLBACK DE STATUS
# ============================================================

def atualizar_status(mensagem):

    status.value = (
        "<b>Status:</b> "
        + html.escape(
            str(mensagem)
        )
    )

    with historico_status:

        print(
            str(mensagem)
        )


# ============================================================
# CALLBACK DO MAPA
#
# Este callback é chamado pelo PlanAppAgent assim que
# o segundo ponto é geocodificado.
# ============================================================

def atualizar_mapa(mapa):

    if mapa is None:

        mapa_output.children = []

        return

    # Atualiza diretamente o VBox.
    #
    # Não usamos display(mapa) aqui.

    mapa_output.children = [
        mapa
    ]


# ============================================================
# AGENTE
# ============================================================

agent = PlanAppAgent(
    progress_callback=atualizar_status,
    map_callback=atualizar_mapa
)


# ============================================================
# EXECUÇÃO ASSÍNCRONA
# ============================================================

async def executar_analise_async():

    texto = entrada.value.strip()

    if not texto:

        status.value = (
            "<b>Status:</b> "
            "Digite uma solicitação."
        )

        return

    # --------------------------------------------------------
    # BLOQUEIA BOTÕES
    # --------------------------------------------------------

    botao_analisar.disabled = True
    botao_nova.disabled = True

    # --------------------------------------------------------
    # LIMPA RESPOSTA E LOG
    # --------------------------------------------------------

    resposta.value = ""

    historico_status.clear_output()

    log_execucao.clear_output()

    mapa_output.children = []

    status.value = (
        "<b>Status:</b> "
        "Iniciando análise..."
    )

    try:

        # ----------------------------------------------------
        # EXECUTA AGENTE
        #
        # O mapa será atualizado pelo callback DURANTE
        # esta chamada, assim que o segundo geocode terminar.
        # ----------------------------------------------------

        resultado = (
            await agent.ask(
                texto
            )
        )

        # ----------------------------------------------------
        # GARANTIA FINAL DO MAPA
        # ----------------------------------------------------

        if agent.map is not None:

            mapa_output.children = [
                agent.map
            ]

        # ----------------------------------------------------
        # RESPOSTA
        # ----------------------------------------------------

        resposta.value = (
            "<div style='"
            "margin-top:15px;"
            "padding:15px;"
            "border:1px solid #ddd;"
            "border-radius:6px;"
            "white-space:pre-wrap;"
            "'>"
            "<b>🤖 Análise do PlanApp AI</b>"
            "<br><br>"
            + html.escape(
                str(resultado)
            ).replace(
                "\n",
                "<br>"
            )
            + "</div>"
        )

        status.value = (
            "<b>Status:</b> "
            "✅ Análise concluída."
        )

    except Exception as exc:

        status.value = (
            "<b>Status:</b> "
            "❌ Erro durante a análise."
        )

        resposta.value = (
            "<div style='"
            "margin-top:15px;"
            "padding:15px;"
            "border:1px solid #d00;"
            "border-radius:6px;"
            "'>"
            "<b>Erro:</b><br>"
            + html.escape(
                str(exc)
            )
            + "</div>"
        )

        with log_execucao:

            print(
                "ERRO:",
                repr(exc)
            )

    finally:

        botao_analisar.disabled = False
        botao_nova.disabled = False


# ============================================================
# CALLBACK DO BOTÃO ANALISAR
# ============================================================

def executar_analise(_):

    asyncio.create_task(
        executar_analise_async()
    )


botao_analisar.on_click(
    executar_analise
)


# ============================================================
# NOVA ANÁLISE
# ============================================================

def nova_analise(_):

    entrada.value = ""

    resposta.value = ""

    mapa_output.children = []

    historico_status.clear_output()

    log_execucao.clear_output()

    status.value = (
        "<b>Status:</b> "
        "Aguardando nova solicitação."
    )


botao_nova.on_click(
    nova_analise
)


# ============================================================
# EXEMPLOS
# ============================================================

exemplos = widgets.HTML(
    value="""
    <div style="
        margin-top:15px;
        padding:10px;
        border:1px solid #ddd;
        border-radius:6px;
    ">
        <b>Exemplos:</b>
        <ul>
            <li>
                Analise um enlace entre a Praça da República
                e o Largo do Paissandu em São Paulo.
            </li>
            <li>
                Analise um enlace entre dois pontos informados
                por endereço.
            </li>
            <li>
                Avalie o perfil e as possíveis obstruções
                de um enlace.
            </li>
        </ul>
    </div>
    """
)


# ============================================================
# INTERFACE
# ============================================================

interface = widgets.VBox(
    [
        titulo,

        entrada,

        widgets.HBox(
            [
                botao_analisar,
                botao_nova
            ]
        ),

        status,

        widgets.HTML(
            value="<b>📋 Histórico de execução</b>"
        ),

        historico_status,

        widgets.HTML(
            value="<b>🛰️ Execução MCP</b>"
        ),

        log_execucao,

        widgets.HTML(
            value=(
                "<b>🗺️ Enlace</b>"
            ),
            layout=widgets.Layout(
                margin="15px 0 5px 0"
            )
        ),

        mapa_output,

        resposta,

        exemplos,
    ],

    layout=widgets.Layout(
        width="100%"
    )
)


display(
    interface
)

In [ ]:
print("==================================================")
print("DIAGNÓSTICO DAS MENSAGENS DO AGENTE")
print("==================================================")

print("\nQuantidade de mensagens:")
print(len(agent.messages))

for i, msg in enumerate(agent.messages):

    print("\n--------------------------------------------------")
    print(f"MENSAGEM {i}")
    print("role:", msg.get("role"))

    if "tool_calls" in msg:

        print("tool_calls:")
        print(msg["tool_calls"])

    content = msg.get("content")

    if content:

        print("content:")
        print(content)

print("\n==================================================")
print("ESTADO FINAL DO AGENTE")
print("==================================================")

print("geocoded_points:")
print(agent.geocoded_points)

print("\nevaluate_executed:")
print(agent.evaluate_executed)

print("\nlast_evaluate_result:")
print(agent.last_evaluate_result)

print("\nmap:")
print(agent.map)